In [ ]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx

# 设置H2分子的几何构型
bond_length = 1.0  # H2平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

# 创建分子对象，使用STO-3G基组
mol = gto.M(atom=geometry, basis='STO-3G')

# 进行Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"Hartree-Fock能量: {E_hf:.8f} Ha")

# 进行FCI计算作为参考
cisolver = fci.FCI(mf)
E_fci, fcivec = cisolver.kernel()
print(f"FCI能量: {E_fci:.8f} Ha")

# 使用NetKet创建哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

In [ ]:
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,  # 总空间轨道数
    s = 1/2,
    n_fermions_per_spin=(1, 1)  # 每种自旋的电子数
)

# 创建采样器 - 使用费米子跳跃采样器
g = nk.graph.Graph(edges=[(0,1),(2,3)])
sa = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

In [ ]:
# 查看所有可能的组态
all_states = hi.all_states()
print("所有可能的组态:")
for i, state in enumerate(all_states):
    occupied = np.where(state == 1)[0]
    print(f"组态 {i}: {state}, 占据轨道: {occupied}")

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
from functools import partial

class CustomConfigWeightModel(nnx.Module):
    """
    自定义组态权重模型
    允许为每个组态指定权重，从而控制组态的比例
    """
    
    def __init__(self, hi, config_weights: dict, *, rngs: nnx.Rngs):
        """
        Args:
            hi: 希尔伯特空间
            config_weights: 组态权重字典，格式为 {tuple(组态): 权重}
                         例如: {(0, 1, 1, 0): 1.0, (1, 0, 1, 0): 0.1}
            rngs: 随机数生成器
        """
        self.hi = hi
        self.config_weights = config_weights
        
        # 将权重转换为可训练参数
        key = rngs.params()
        
        # 为每个组态创建一个可训练的振幅参数
        n_configs = len(config_weights)
        self.amplitudes = nnx.Param(
            jnp.array([config_weights[tuple(config)] for config in config_weights.keys()], dtype=jnp.float32)
        )
        
        # 存储组态列表用于匹配
        self.config_list = list(config_weights.keys())
    
    def __call__(self, n: jax.Array) -> jax.Array:
        @partial(jnp.vectorize, signature="(n)->()")
        def log_psi(n):
            # 找到匹配的组态
            n_tuple = tuple(n)
            
            # 查找匹配的组态索引
            idx = -1
            for i, config in enumerate(self.config_list):
                if config == n_tuple:
                    idx = i
                    break
            
            if idx >= 0:
                # 返回对数振幅
                return jnp.log(self.amplitudes[idx])
            else:
                # 如果组态不在列表中，返回负无穷（概率为0）
                return -jnp.inf
        
        return log_psi(n)


class NeuralConfigWeightModel(nnx.Module):
    """
    神经网络组态权重模型
    使用神经网络学习组态权重，初始权重可以自定义
    """
    
    def __init__(self, hi, config_weights: dict, hidden_units: int = 8, *, rngs: nnx.Rngs):
        """
        Args:
            hi: 希尔伯特空间
            config_weights: 组态权重字典，格式为 {tuple(组态): 权重}
            hidden_units: 隐藏层单元数
            rngs: 随机数生成器
        """
        self.hi = hi
        self.config_list = list(config_weights.keys())
        
        # 使用神经网络学习组态权重
        key = rngs.params()
        
        # 初始权重
        initial_weights = jnp.array([config_weights[tuple(config)] for config in config_weights.keys()], dtype=jnp.float32)
        
        # 创建神经网络
        self.linear1 = nnx.Linear(in_features=hi.size, out_features=hidden_units, rngs=rngs)
        self.linear2 = nnx.Linear(in_features=hidden_units, out_features=len(config_weights), rngs=rngs)
        
        # 偏置项初始化为对数初始权重
        self.bias = nnx.Param(jnp.log(initial_weights))
    
    def __call__(self, n: jax.Array) -> jax.Array:
        @partial(jnp.vectorize, signature="(n)->()")
        def log_psi(n):
            # 找到匹配的组态
            n_tuple = tuple(n)
            
            # 查找匹配的组态索引
            idx = -1
            for i, config in enumerate(self.config_list):
                if config == n_tuple:
                    idx = i
                    break
            
            if idx >= 0:
                # 通过神经网络计算权重
                x = n.astype(jnp.float32)
                h = jax.nn.tanh(self.linear1(x))
                log_weights = self.linear2(h) + self.bias
                
                # 返回对应组态的对数振幅
                return log_weights[idx]
            else:
                # 如果组态不在列表中，返回负无穷（概率为0）
                return -jnp.inf
        
        return log_psi(n)

In [ ]:
# 定义自定义组态权重
# 键是组态（元组格式），值是权重（相对比例）
# H2分子在STO-3G基组下有4个自旋轨道，2个电子
# 主要组态:
# [0, 1, 1, 0]: 电子占据轨道1和2（Hartree-Fock组态）
# [1, 0, 1, 0]: 电子占据轨道0和2
# [0, 1, 0, 1]: 电子占据轨道1和3
# [1, 0, 0, 1]: 电子占据轨道0和3

config_weights = {
    (0, 1, 1, 0): 1.0,  # Hartree-Fock组态，权重最大
    (1, 0, 1, 0): 0.1,  # 其他组态权重较小
    (0, 1, 0, 1): 0.1,
    (1, 0, 0, 1): 0.1,
}

print("自定义组态权重:")
for config, weight in config_weights.items():
    occupied = np.where(np.array(config) == 1)[0]
    print(f"组态 {config}, 占据轨道: {occupied}, 权重: {weight}")

In [ ]:
# 创建模型 - 使用固定权重模型
model = CustomConfigWeightModel(hi, config_weights, rngs=nnx.Rngs(2))

# 创建MCState
vs = nk.vqs.MCState(sa, model, n_discard_per_chain=10, n_samples=512)

# 设置优化器
opt = nk.optimizer.Sgd(learning_rate=0.05)
sr = nk.optimizer.SR(diag_shift=0.01)

# 创建VMC驱动器
gs = nk.driver.VMC(ha, opt, variational_state=vs, preconditioner=sr)

# 运行优化
exp_name = "h2_molecule_custom_config_fixed"

In [ ]:
gs.run(300, out=exp_name)

In [ ]:
# 读取日志数据并绘图
with open(f"{exp_name}.log") as f:
    data = json.load(f)

x = data["Energy"]["iters"]
y = data["Energy"]["Mean"]

# 绘制能量收敛曲线
plt.figure(figsize=(10, 6))
plt.axhline(E_fci, color="red", linestyle="--", label=f"FCI energy = {E_fci:.6f} Ha")
plt.axhline(E_hf, color="green", linestyle="--", label=f"Hartree Fock energy = {E_hf:.6f} Ha")
plt.plot(x, y, 'b-', label="VMC energy (fixed weights)")
plt.xlabel("Iterations")
plt.ylabel("Energy (Ha)")
plt.title("H2 molecule energy convergence - Custom Config Weights (Fixed)")
plt.legend()
plt.grid(True)
plt.show()

print(f"\n最终VMC能量: {y[-1]:.8f} Ha")
print(f"与FCI能量误差: {abs(y[-1] - E_fci):.8f} Ha")

In [ ]:
# 查看组态概率分布
n_samples = 10000
samples = vs.sample(n_samples=n_samples)
samples_flat = samples.reshape(-1, samples.shape[-1])

# 统计每个组态的出现次数
unique_configs, counts = np.unique(samples_flat, axis=0, return_counts=True)
probabilities = counts / np.sum(counts)

# 按概率排序
sorted_indices = np.argsort(probabilities)[::-1]
sorted_configs = unique_configs[sorted_indices]
sorted_probs = probabilities[sorted_indices]

print("组态及其概率分布 (固定权重模型):")
print("组态(轨道占据)    概率      初始权重")
print("-" * 50)
for i, (config, prob) in enumerate(zip(sorted_configs, sorted_probs)):
    config_tuple = tuple(config)
    weight = config_weights.get(config_tuple, 0.0)
    occupied = np.where(config == 1)[0]
    print(f"{config}         {prob:.6f}      {weight:.1f}")

In [ ]:
# 使用神经网络模型 - 允许权重在训练过程中调整
neural_model = NeuralConfigWeightModel(hi, config_weights, hidden_units=8, rngs=nnx.Rngs(3))

# 创建新的MCState
vs_neural = nk.vqs.MCState(sa, neural_model, n_discard_per_chain=10, n_samples=512)

# 设置优化器
opt = nk.optimizer.Sgd(learning_rate=0.05)
sr = nk.optimizer.SR(diag_shift=0.01)

# 创建VMC驱动器
gs_neural = nk.driver.VMC(ha, opt, variational_state=vs_neural, preconditioner=sr)

# 运行优化
exp_name_neural = "h2_molecule_custom_config_neural"

In [ ]:
gs_neural.run(300, out=exp_name_neural)

In [ ]:
# 读取神经网络模型的日志数据
with open(f"{exp_name_neural}.log") as f:
    data_neural = json.load(f)

x_neural = data_neural["Energy"]["iters"]
y_neural = data_neural["Energy"]["Mean"]

# 绘制两种模型的对比
plt.figure(figsize=(12, 6))
plt.axhline(E_fci, color="red", linestyle="--", label=f"FCI energy = {E_fci:.6f} Ha")
plt.axhline(E_hf, color="green", linestyle="--", label=f"Hartree Fock energy = {E_hf:.6f} Ha")
plt.plot(x, y, 'b-', label="VMC energy (fixed weights)")
plt.plot(x_neural, y_neural, 'r-', label="VMC energy (neural weights)")
plt.xlabel("Iterations")
plt.ylabel("Energy (Ha)")
plt.title("H2 molecule energy convergence - Comparison")
plt.legend()
plt.grid(True)
plt.show()

print(f"\n固定权重模型最终能量: {y[-1]:.8f} Ha, 误差: {abs(y[-1] - E_fci):.8f} Ha")
print(f"神经网络模型最终能量: {y_neural[-1]:.8f} Ha, 误差: {abs(y_neural[-1] - E_fci):.8f} Ha")

In [ ]:
# 比较两种模型的组态概率分布
n_samples = 10000
samples_neural = vs_neural.sample(n_samples=n_samples)
samples_neural_flat = samples_neural.reshape(-1, samples_neural.shape[-1])

# 统计神经网络模型的组态概率
unique_configs_neural, counts_neural = np.unique(samples_neural_flat, axis=0, return_counts=True)
probabilities_neural = counts_neural / np.sum(counts_neural)

# 创建概率对比表
print("组态概率对比:")
print("组态          固定权重模型    神经网络模型    初始权重")
print("-" * 60)
for config in config_weights.keys():
    config_arr = np.array(config)
    
    # 查找固定权重模型的概率
    idx_fixed = np.where((sorted_configs == config_arr).all(axis=1))[0]
    prob_fixed = sorted_probs[idx_fixed[0]] if len(idx_fixed) > 0 else 0.0
    
    # 查找神经网络模型的概率
    idx_neural = np.where((unique_configs_neural == config_arr).all(axis=1))[0]
    prob_neural = probabilities_neural[idx_neural[0]] if len(idx_neural) > 0 else 0.0
    
    occupied = np.where(config_arr == 1)[0]
    print(f"{config}    {prob_fixed:.6f}      {prob_neural:.6f}      {config_weights[config]:.1f}")

In [ ]:
# 测试不同的权重配置
test_configs = [
    # 只包含Hartree-Fock组态
    {(0, 1, 1, 0): 1.0},
    
    # 均匀权重
    {(0, 1, 1, 0): 1.0, (1, 0, 1, 0): 1.0, (0, 1, 0, 1): 1.0, (1, 0, 0, 1): 1.0},
    
    # 强调双激发组态
    {(0, 1, 1, 0): 0.5, (1, 0, 1, 0): 0.5, (0, 1, 0, 1): 0.5, (1, 0, 0, 1): 2.0},
]

test_results = []

for i, test_config in enumerate(test_configs):
    print(f"\n测试配置 {i+1}:")
    for config, weight in test_config.items():
        occupied = np.where(np.array(config) == 1)[0]
        print(f"  组态 {config}, 占据轨道: {occupied}, 权重: {weight}")
    
    # 创建模型
    model_test = CustomConfigWeightModel(hi, test_config, rngs=nnx.Rngs(10+i))
    vs_test = nk.vqs.MCState(sa, model_test, n_discard_per_chain=10, n_samples=512)
    
    # 设置优化器
    opt = nk.optimizer.Sgd(learning_rate=0.05)
    sr = nk.optimizer.SR(diag_shift=0.01)
    
    # 创建VMC驱动器
    gs_test = nk.driver.VMC(ha, opt, variational_state=vs_test, preconditioner=sr)
    
    # 运行优化
    exp_name_test = f"h2_molecule_test_config_{i+1}"
    gs_test.run(200, out=exp_name_test)
    
    # 读取结果
    with open(f"{exp_name_test}.log") as f:
        data_test = json.load(f)
    
    final_energy = data_test["Energy"]["Mean"][-1]
    test_results.append(final_energy)
    
    print(f"  最终能量: {final_energy:.8f} Ha, 误差: {abs(final_energy - E_fci):.8f} Ha")

In [ ]:
# 绘制所有测试结果的对比
plt.figure(figsize=(12, 6))

# 绘制参考线
plt.axhline(E_fci, color="red", linestyle="--", label=f"FCI energy = {E_fci:.6f} Ha")
plt.axhline(E_hf, color="green", linestyle="--", label=f"Hartree Fock energy = {E_hf:.6f} Ha")

# 绘制不同配置的结果
labels = ['Hartree-Fock only', 'Uniform weights', 'Double excitation emphasized']
colors = ['b', 'g', 'm']

for i, exp_name in enumerate(["h2_molecule_test_config_1", "h2_molecule_test_config_2", "h2_molecule_test_config_3"]):
    with open(f"{exp_name}.log") as f:
        data_test = json.load(f)
    
    x_test = data_test["Energy"]["iters"]
    y_test = data_test["Energy"]["Mean"]
    
    plt.plot(x_test, y_test, color=colors[i], label=f"{labels[i]} (final: {y_test[-1]:.6f} Ha)")

plt.xlabel("Iterations")
plt.ylabel("Energy (Ha)")
plt.title("H2 molecule energy convergence - Different Config Weight Strategies")
plt.legend()
plt.grid(True)
plt.show()

print("\n不同权重配置的最终能量对比:")
print("-" * 60)
print(f"{'配置':<30} {'能量 (Ha)':<15} {'误差 (Ha)':<15}")
print("-" * 60)
print(f"{'FCI':<30} {E_fci:<15.8f} {0:<15.8f}")
print(f"{'Hartree-Fock':<30} {E_hf:<15.8f} {abs(E_hf - E_fci):<15.8f}")
for i, (label, energy) in enumerate(zip(labels, test_results)):
    print(f"{label:<30} {energy:<15.8f} {abs(energy - E_fci):<15.8f}")

In [ ]:
# 最终总结
print("=" * 70)
print("自定义组态权重模型总结")
print("=" * 70)
print()
print("本notebook实现了自定义组态权重的H2分子VMC计算，主要特点:")
print()
print("1. CustomConfigWeightModel: 固定权重模型")
print("   - 用户可以指定每个组态的权重")
print("   - 权重在训练过程中保持不变")
print("   - 适用于已知物理直觉的情况")
print()
print("2. NeuralConfigWeightModel: 神经网络权重模型")
print("   - 初始权重由用户指定")
print("   - 神经网络在训练过程中学习最优权重")
print("   - 具有更强的表达能力")
print()
print("3. 测试了不同的权重配置策略:")
print("   - Hartree-Fock only: 只包含参考组态")
print("   - Uniform weights: 所有组态权重相同")
print("   - Double excitation emphasized: 强调双激发组态")
print()
print("通过自定义组态权重，可以:")
print("- 探索不同组态对能量的贡献")
print("- 验证物理直觉")
print("- 加速收敛")
print("- 理解电子关联效应")